# Description

Predicts drug-disease associations using the **gene-based** approach: raw gene-level z-scores from SMulTiXcan (disease) and LINCS L1000 (drug), without projecting into the CLAMP latent space.

The prediction score for a drug-disease pair is:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$

computed on the intersection of genes present in both LINCS and SMulTiXcan.

This follows the framework of [Menden et al., 2020 (Nature Neuroscience)](https://doi.org/10.1038/nn.4618).

Results are saved as HDF5 files with keys:
- `full_prediction`: predictions for all traits
- `prediction`: predictions mapped to DOID (for comparison with PharmacotherapyDB gold standard)
- `metadata`: method name, n_top_genes, data source

# Module loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [3]:
# if True, re-run even if output files already exist
FORCE_RUN = True

PREDICTION_METHOD = 'Gene-based'

In [4]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

PROJECTIONS_DIR = here('output/drug_disease_analyses')
display(PROJECTIONS_DIR)
assert PROJECTIONS_DIR.exists()

OUTPUT_PREDICTIONS_DIR = PROJECTIONS_DIR / 'predictions' / 'dotprod_neg'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg')

# Helper functions

In [5]:
def map_traits_to_doid(data, preferred_doids, ukb_efo, efo_xrefs, do_xrefs):
    """
    Maps trait columns (UKB full codes) to Disease Ontology IDs (DOID).

    For traits mapping to multiple DOIDs, prefers those in `preferred_doids`
    (DOIDs present in the gold standard). When a DOID appears from multiple
    traits, keeps the maximum score.

    Args:
        data: DataFrame with traits as columns (UKB full-code format).
        preferred_doids: set of DOID strings present in the gold standard.
        ukb_efo: DataFrame indexed by ukb_fullcode, with column 'term_codes'
                 (comma-separated EFO codes).
        efo_xrefs: DataFrame with columns ['term_id', 'target_id_type', 'target_id']
                   (maps EFO IDs to DOID via exact EFO:XXXXXXX format).
        do_xrefs: DataFrame with columns ['doid_code', 'resource', 'resource_id']
                  (maps EFO numbers without 'EFO:' prefix to DOID codes).

    Returns:
        DataFrame with DOID columns. Traits without a DOID mapping are dropped.
        When multiple traits map to the same DOID, maximum score is kept.
    """
    doid_efo = efo_xrefs[efo_xrefs['target_id_type'] == 'DOID']

    trait_to_doid = {}
    for trait in data.columns:
        if trait not in ukb_efo.index:
            continue
        rows = ukb_efo.loc[trait]
        if isinstance(rows, pd.Series):
            rows = rows.to_frame().T

        all_efo_codes = set()
        for term_codes in rows['term_codes'].dropna():
            for code in str(term_codes).split(','):
                code = code.strip()
                if code:
                    all_efo_codes.add(code)

        all_doids = set()
        for efo_code in all_efo_codes:
            mask = doid_efo['term_id'] == efo_code
            all_doids.update(doid_efo[mask]['target_id'].values)
            if efo_code.startswith('EFO:'):
                efo_num = efo_code[4:]
                mask2 = (do_xrefs['resource'] == 'EFO') & (do_xrefs['resource_id'] == efo_num)
                all_doids.update(do_xrefs[mask2]['doid_code'].values)

        if not all_doids:
            continue

        preferred = sorted(all_doids & preferred_doids)
        trait_to_doid[trait] = preferred[0] if preferred else sorted(all_doids)[0]

    data_mapped = data.loc[:, list(trait_to_doid.keys())].rename(columns=trait_to_doid)
    data_mapped = data_mapped.T.groupby(level=0).max().T
    return data_mapped

In [6]:
def zero_nontop_genes(trait_vector, n_top, use_abs=True):
    """Zeros all but the top `n_top` gene values in a Series."""
    values = trait_vector.abs() if use_abs else trait_vector
    top_idx = values.sort_values(ascending=False).head(n_top).index
    result = trait_vector.copy()
    result[~result.index.isin(top_idx)] = 0.0
    return result

In [7]:
def predict_and_save(
    lincs_data,
    smultixcan_data,
    output_dir,
    doids_in_gold_standard,
    trait_to_doid_func,
    method_name,
    data_stem,
    n_top=None,
    use_abs=True,
    force_run=True,
):
    """
    Computes dot-product drug-disease predictions and saves to HDF5.
    score = -1 * drug^T * disease  (on intersection of genes)
    """
    suffix = 'all_genes' if n_top is None else f'top_{n_top}_genes'
    output_file = output_dir / f'{data_stem}-{suffix}-prediction_scores.h5'

    print(f'predicting {suffix}...', end='')
    if output_file.exists() and not force_run:
        print('  already run')
        return
    print('')

    disease_data = smultixcan_data.copy()
    if n_top is not None:
        disease_data = disease_data.apply(lambda x: zero_nontop_genes(x, n_top, use_abs))

    # score = -1 * (genes x drugs)^T dot (genes x traits) → drugs x traits
    scores = -1.0 * lincs_data.T.dot(disease_data)
    print(f'  shape: {scores.shape}')

    with pd.HDFStore(output_file, mode='w', complevel=4) as store:
        # full prediction
        scores.index.name = 'drug'
        scores.columns.name = 'trait'
        full_pred = (
            scores.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        full_pred['trait'] = full_pred['trait'].astype('category')
        full_pred['drug'] = full_pred['drug'].astype('category')
        assert full_pred.shape == full_pred.dropna().shape
        print(f'  full_prediction shape: {full_pred.shape}')
        display(full_pred.describe())
        store.put('full_prediction', full_pred, format='table')

        # DOID-mapped prediction
        scores_doid = trait_to_doid_func(scores)
        print(f'  shape after DOID map: {scores_doid.shape}')
        assert scores_doid.index.is_unique
        assert scores_doid.columns.is_unique

        scores_doid.index.name = 'drug'
        scores_doid.columns.name = 'trait'
        doid_pred = (
            scores_doid.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        doid_pred['trait'] = doid_pred['trait'].astype('category')
        doid_pred['drug'] = doid_pred['drug'].astype('category')
        assert doid_pred.shape == doid_pred.dropna().shape
        print(f'  prediction shape: {doid_pred.shape}')
        store.put('prediction', doid_pred, format='table')

        meta = pd.DataFrame({
            'method': [method_name],
            'n_top_genes': [-1.0 if n_top is None else float(n_top)],
            'data': [data_stem],
        })
        store.put('metadata', meta, format='table')

    print(f'  saved to: {output_file}')

# Load PharmacotherapyDB gold standard

In [8]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait→DOID mapping files

In [9]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
display(ukb_efo.shape)
display(ukb_efo.head())

(1087, 6)

,ukb_code,term_label,term_codes,mapping_type,current_term_label,category
ukb_fullcode,,,,,,
K55-Diagnoses_main_ICD10_K55_Vascular_disorders_of_intestine,K55,vascular disease,"EFO:0004264, EFO:0009431",Broad,vascular disease AND intestinal disease,disease
M17-Diagnoses_main_ICD10_M17_Gonarthrosis_arthrosis_of_knee,M17,osteoarthritis || knee,EFO:0004616,Broad,"osteoarthritis, knee",disease
R30-Diagnoses_main_ICD10_R30_Pain_associated_with_micturition,R30,dysuria,EFO:0003901,? Broad,dysuria,NaN
O60-Diagnoses_main_ICD10_O60_Preterm_delivery,O60,premature birth,EFO:0003917,? Exact,premature birth,NaN
S64-Diagnoses_main_ICD10_S64_Injury_of_nerves_at_wrist_and_hand_level,S64,carpal tunnel syndrome,EFO:0004143,? Narrow,carpal tunnel syndrome,disease


In [10]:
efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
display(efo_xrefs.shape)

(104094, 3)

In [11]:
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')
display(do_xrefs.shape)

(8204, 4)

In [12]:
def trait_to_doid_func(data):
    return map_traits_to_doid(data, doids_in_gold_standard, ukb_efo, efo_xrefs, do_xrefs)

# Load raw gene-level data

In [13]:
lincs_raw = pd.read_pickle(DATA_DIR / 'lincs-data.pkl')
print(f'LINCS raw shape: {lincs_raw.shape}')
display(lincs_raw.head())

LINCS raw shape: (7120, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
ENSG00000196839,-1.001,-1.835,1.391,1.132,0.257,1.932,0.508,1.408,0.777,0.032,...,-1.692,-0.516,-1.435,-0.317,-0.012,0.641,-0.230,-0.518,-0.177,2.146
ENSG00000170558,1.146,-1.863,0.011,-1.020,1.143,-0.115,1.327,0.310,-1.853,0.872,...,0.354,0.498,0.268,-1.084,-0.142,-0.077,0.633,-1.807,0.032,0.135
ENSG00000117020,-0.693,1.694,-0.804,-0.164,1.145,-1.465,1.221,-0.747,0.829,-0.961,...,-1.196,-0.230,-1.049,-0.347,0.586,0.865,-0.021,2.180,-0.956,0.105
ENSG00000133997,-0.037,0.383,0.269,-0.997,0.185,-0.536,0.424,-0.119,-1.313,0.579,...,-0.343,0.116,-0.245,-0.127,-1.367,0.149,0.117,2.084,1.178,0.772
ENSG00000101473,0.162,-0.899,0.105,-0.090,-1.291,1.404,0.185,0.157,-0.327,-0.026,...,-0.136,-1.115,-0.280,0.200,0.638,-0.197,-0.360,-2.302,-0.117,-0.167


In [14]:
smultixcan_raw = pd.read_pickle(DATA_DIR / 'smultixcan-mashr-zscores.pkl')
print(f'SMulTiXcan raw shape: {smultixcan_raw.shape}')
display(smultixcan_raw.head())

SMulTiXcan raw shape: (22515, 4091)


,20096_1-Size_of_red_wine_glass_drunk_small_125ml,2345-Ever_had_bowel_cancer_screening,N49-Diagnoses_main_ICD10_N49_Inflammatory_disorders_of_male_genital_organs_not_elsewhere_classified,100011_raw-Iron,5221-Index_of_best_refractometry_result_right,20003_1141150624-Treatmentmedication_code_zomig_25mg_tablet,S69-Diagnoses_main_ICD10_S69_Other_and_unspecified_injuries_of_wrist_and_hand,20024_1136-Job_code_deduced_Information_and_communication_technology_managers,20002_1385-Noncancer_illness_code_selfreported_allergy_or_anaphylactic_reaction_to_food,G6_SLEEPAPNO-Sleep_apnoea,...,Astle_et_al_2016_Sum_basophil_neutrophil_counts,RA_OKADA_TRANS_ETHNIC,pgc.scz2,PGC_ADHD_EUR_2017,MAGIC_FastingGlucose,Astle_et_al_2016_Red_blood_cell_count,SSGAC_Depressive_Symptoms,BCAC_ER_positive_BreastCancer_EUR,IBD.EUR.Inflammatory_Bowel_Disease,Astle_et_al_2016_High_light_scatter_reticulocyte_count
gene_name,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.169468,0.102558,0.239545,0.887758,1.313448,1.472148,0.726160,1.516367,1.299771,1.068093,...,0.813014,0.275993,0.510834,0.024717,0.430951,0.824314,0.367414,1.377624,0.738444,0.298259
ENSG00000000457,1.358856,1.846875,0.139324,0.129530,0.757757,1.103979,0.612418,1.822327,2.035372,1.008058,...,1.441795,0.654791,2.545653,1.202984,0.514244,0.237223,0.414171,0.101731,1.012735,0.945167
ENSG00000000460,0.151008,1.173202,1.179426,0.571656,0.098771,0.221072,0.276415,0.461381,0.855502,0.201876,...,0.668962,0.300040,0.541782,1.033308,0.482261,0.695624,0.336480,0.083316,3.493196,0.991948
ENSG00000000938,1.302722,0.841524,1.578926,0.721340,0.139314,4.387016,0.125959,1.247123,0.215124,0.892083,...,0.126657,0.048048,1.886356,0.540496,0.127524,1.494501,0.056432,1.704863,1.351619,1.027297
ENSG00000000971,1.338813,0.262339,0.689379,1.702019,0.325859,0.063161,1.141126,0.882682,0.035533,1.810191,...,0.858497,1.675562,2.319072,1.598721,0.162958,0.005703,3.004544,0.803669,0.444266,0.165671


In [15]:
# Drop traits with all-NaN gene associations; fill remaining NaNs with 0
n_before = smultixcan_raw.shape[1]
smultixcan_raw = smultixcan_raw.dropna(axis=1, how='all').fillna(0.0)
print(f'Traits after dropping all-NaN columns: {smultixcan_raw.shape[1]} (was {n_before})')

Traits after dropping all-NaN columns: 4091 (was 4091)


In [16]:
# Intersect genes present in both LINCS and SMulTiXcan
common_genes = lincs_raw.index.intersection(smultixcan_raw.index)
print(f'Common genes (LINCS ∩ SMulTiXcan): {len(common_genes)}')

lincs_common = lincs_raw.loc[common_genes]
smultixcan_common = smultixcan_raw.loc[common_genes]

print(f'LINCS (common genes) shape: {lincs_common.shape}')
print(f'SMulTiXcan (common genes) shape: {smultixcan_common.shape}')

Common genes (LINCS ∩ SMulTiXcan): 7120
LINCS (common genes) shape: (7120, 1170)
SMulTiXcan (common genes) shape: (7120, 4091)


# Predict drug-disease associations

In [17]:
# Top-N gene thresholds (None = use all genes in common)
N_TOP_GENES_LIST = [None, 50, 100, 250, 500]

DATA_STEM = 'smultixcan-mashr-zscores-data'

for n_top in N_TOP_GENES_LIST:
    predict_and_save(
        lincs_data=lincs_common,
        smultixcan_data=smultixcan_common,
        output_dir=OUTPUT_PREDICTIONS_DIR,
        doids_in_gold_standard=doids_in_gold_standard,
        trait_to_doid_func=trait_to_doid_func,
        method_name=PREDICTION_METHOD,
        data_stem=DATA_STEM,
        n_top=n_top,
        use_abs=True,
        force_run=FORCE_RUN,
    )
    print()

predicting all_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,2.394107e+01
std,6.659231e+02
min,-2.929976e+04
25%,-2.700084e+02
50%,-6.660593e+00
75%,2.990221e+02
max,1.285924e+04


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-data-all_genes-prediction_scores.h5

predicting top_50_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,-3.771793e+00
std,1.115039e+02
min,-2.580067e+04
25%,-1.785317e+01
50%,-1.282735e-01
75%,1.753727e+01
max,8.987540e+03


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-data-top_50_genes-prediction_scores.h5

predicting top_100_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


...

  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-data-top_100_genes-prediction_scores.h5

predicting top_250_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,-4.884954e+00
std,1.897841e+02
min,-3.962712e+04
25%,-4.202816e+01
50%,-2.777296e-01
75%,4.257752e+01
max,7.551945e+03


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-data-top_250_genes-prediction_scores.h5

predicting top_500_genes...
  shape: (1170, 4091)
  full_prediction shape: (4786470, 3)


,score
count,4.786470e+06
mean,-3.662821e+00
std,2.363408e+02
min,-4.462193e+04
25%,-6.489313e+01
50%,-8.492873e-01
75%,6.668651e+01
max,6.756465e+03


  shape after DOID map: (1170, 364)
  prediction shape: (425880, 3)
  saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg/smultixcan-mashr-zscores-data-top_500_genes-prediction_scores.h5

